# Introducing Delivery Exception Desk

When a parcel goes missing or returns to the warehouse, a support team has to decide what happens next. Delivery Exception Desk uses the customer message, order details, carrier evidence, and delivery policy to propose that action.

1. Explore the bilingual case data.
2. Ask a free model to make a decision in one call.
3. Give a LangChain agent tools to retrieve evidence and record its decision.

All cases and decisions are synthetic. This notebook simulates actions in memory.

## Setup

Run `uv sync` from the project root. Open the notebook in a Jupyter-compatible editor and select the project's `.venv` Python kernel.

Part 1 runs locally. Parts 2 and 3 read `GROQ_API_KEY` from your environment or a local `.env` at the project root.

The default sample runs five working-queue cases with GPT-OSS 20B on Groq. Set `tiny = False` to run the whole selected set.

[GPT-OSS 20B on Groq](https://console.groq.com/docs/model/openai/gpt-oss-20b)

## 1. Examine the data

The working queue is the development set. Use it to inspect cases and iterate on the agent. Use `held_out` once the prompt is fixed. The `harder` and `even_harder` sets exercise more ambiguous evidence.

Each ticket has a customer message, an order, and carrier scans. The answer key records the analyst's expected action, refund, policy reference, and rationale.

- **Routine** cases follow a straightforward rule.
- **Hard** cases involve timing, ambiguous evidence, or rules that interact.
- **Manual** cases also need verified resolution history. The label's `needs` field names the extra evidence.

Amounts are in Saudi riyals (SAR). The **requested refund** is the amount the customer asks for. An unconditional refund request with no stated amount uses the order total. If there is no unconditional monetary request, the amount is missing. The **expected refund** is what the analyst would authorize under the policy, or zero for other actions.

In [1]:
import json
from collections import Counter
from pathlib import Path

import plotly.graph_objects as go
from IPython.display import Markdown, display

dataset = "queue"  # queue, held_out, harder, or even_harder
tiny = True

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
data_dir = root / "data"
policy = (root / "docs" / "delivery-exception-policy.md").read_text(encoding="utf-8")


def load_json(relative_path):
    return json.loads((data_dir / relative_path).read_text(encoding="utf-8"))


history_path = "prior_resolutions.json"
if dataset == "queue":
    tickets = load_json("cases.json")
    orders = {row["order_id"]: row for row in load_json("orders.json")}
    shipments = {row["tracking_id"]: row for row in load_json("shipments.json")}
    cases = []
    for ticket in tickets:
        order = orders[ticket["order_id"]]
        shipment = shipments[ticket["tracking_id"]]
        assert shipment["order_id"] == order["order_id"]
        cases.append({**ticket, "order": order, "shipment": shipment})
    labels = load_json("evals/decisions.json")
elif dataset == "held_out":
    cases = load_json("evals/held_out_cases.json")
    labels = load_json("evals/held_out_decisions.json")
elif dataset in {"harder", "even_harder"}:
    cases = load_json(f"evals/{dataset}/cases.json")
    labels = load_json(f"evals/{dataset}/decisions.json")
    history_path = "evals/even_harder/resolution_history.json" if dataset == "even_harder" else None
else:
    raise ValueError("Choose queue, held_out, harder, or even_harder.")

case_ids = {case["case_id"] for case in cases}
gold_by_id = {label["case_id"]: label for label in labels}
history = load_json(history_path) if history_path else []
history_by_id = {entry["case_id"]: entry for entry in history if entry["case_id"] in case_ids}

assert len(cases) == len(case_ids) == len(labels) == len(gold_by_id)
assert case_ids == gold_by_id.keys()
assert {label["case_id"] for label in labels if label["difficulty"] == "manual"} == history_by_id.keys()

sample = [
    case for case in cases
    if not tiny or gold_by_id[case["case_id"]]["tiny"]
]

print(f"{dataset}: {len(cases)} cases, {len(sample)} selected")
print("Languages:", dict(Counter(case["language"] for case in cases)))
print("Exception types:", dict(Counter(case["exception_type"] for case in cases)))

queue: 24 cases, 5 selected
Languages: {'ar': 12, 'en': 12}
Exception types: {'delivered_not_received': 12, 'return_to_sender': 12}


In [2]:
def inspect_case(case):
    case_id = case["case_id"]
    label = gold_by_id[case_id]
    requested = case["requested_refund_sar"]
    requested_text = f"SAR {requested:,.2f}" if requested is not None else "No amount requested"

    print(f'{case_id} ({case["language"]}, {case["exception_type"]}, {label["difficulty"]}), {case["reported_at"]}')
    print(case["customer_message"])
    print(f'Order value: SAR {case["order"]["total_sar"]:,.2f} | Requested: {requested_text}')
    print("Carrier events:")
    for event in case["shipment"]["events"]:
        print(f'  {event["at"]} | {event["status"]} | {event["detail"]}')
    print("Proof:", case["shipment"]["proof_of_delivery"])
    if label["needs"]:
        print("Resolution history:", history_by_id[case_id])
    print(f'Expected: {label["action"]} | Refund: SAR {label["refund_sar"]:,.2f}')
    print(f'{label["policy_ref"]}: {label["rationale"]}')


example = sample[0]
manual_case = next((case for case in cases if case["case_id"] in history_by_id), None)
refund_example = next((case for case in cases if case["requested_refund_sar"] is not None), None)
examples = {case["case_id"]: case for case in (example, refund_example, manual_case) if case}
for case in examples.values():
    inspect_case(case)
    print()

DED-2001 (ar, delivered_not_received, routine), 2026-10-07T15:12:00+03:00
جاني إشعار إن سلة الغسيل وصلت للبيت قبل الظهر. عندنا مدخل للعائلة ومدخل للمجلس وشيكت الاثنين، ما حصلتها. التتبع ما فيه أي صورة.
Order value: SAR 137.50 | Requested: No amount requested
Carrier events:
  2026-10-07T08:18:00+03:00 | out_for_delivery | Parcel was assigned to a van route because of its size.
  2026-10-07T11:40:00+03:00 | delivered | Driver closed the stop as a drop-off at the dwelling entrance.
Proof: None
Expected: wait_for_update | Refund: SAR 0.00
P2: The delivery scan is 3 hours 32 minutes old and there is no proof of a mismatch.

DED-2003 (ar, return_to_sender, hard), 2026-10-08T10:35:00+03:00
صباح الخير. المراية رجعت للمستودع حسب آخر تحديث. كنت طالبها للمحل واضطريت أركّب وحدة ثانية. أبي ترجعون ٥٢٠ ريال لو سمحتوا، مبلغ الطلب ومعه ٣٠ ريال دفعتها أجرة سيارة عشان أروح للفرع.
Order value: SAR 490.00 | Requested: SAR 520.00
Carrier events:
  2026-10-02T17:44:00+03:00 | delivery_attempted | Recipient 

### Case mix and refund amounts

Which actions are common in each difficulty group? Find a case where the amount requested differs from the expected refund and check why.

In [3]:
exception_names = {
    "delivered_not_received": "Delivered but missing",
    "return_to_sender": "Return to sender",
}
mix = Counter((case["language"], case["exception_type"]) for case in cases)
mix_chart = go.Figure()
for language, name in (("ar", "Arabic"), ("en", "English")):
    mix_chart.add_bar(
        name=name,
        x=list(exception_names.values()),
        y=[mix[language, exception] for exception in exception_names],
    )
mix_chart.update_layout(title="Cases by exception type and language", barmode="group", yaxis_title="Cases")
mix_chart.show()

difficulties = ["routine", "hard", "manual"]
difficulty_counts = Counter((label["difficulty"], label["action"]) for label in labels)
difficulty_chart = go.Figure()
for action in sorted({label["action"] for label in labels}):
    difficulty_chart.add_bar(
        name=action.replace("_", " "),
        x=difficulties,
        y=[difficulty_counts[difficulty, action] for difficulty in difficulties],
    )
difficulty_chart.update_layout(title="Expected actions by difficulty", barmode="stack", yaxis_title="Cases")
difficulty_chart.show()

refund_cases = [
    case for case in cases
    if case["requested_refund_sar"] is not None or gold_by_id[case["case_id"]]["refund_sar"] > 0
]
refund_chart = go.Figure()
refund_chart.add_bar(
    name="Customer requested",
    x=[case["case_id"] for case in refund_cases],
    y=[case["requested_refund_sar"] for case in refund_cases],
)
refund_chart.add_bar(
    name="Analyst expected",
    x=[case["case_id"] for case in refund_cases],
    y=[gold_by_id[case["case_id"]]["refund_sar"] for case in refund_cases],
)
refund_chart.update_layout(
    title="Requested and expected refunds by case",
    barmode="group",
    xaxis={"type": "category", "title": "Case"},
    yaxis_title="Amount (SAR)",
)
refund_chart.show()

## 2. Run with a free model

Send the ticket, order, carrier evidence, and policy in one request. Structured output constrains the action, refund, review flag, policy reference, and rationale.

The baseline has no verified resolution history. For manual cases, it applies the rules it can evaluate from the evidence it receives. Iterate on the working queue. Keep the prompt fixed when you switch to an evaluation set.

Requests run one at a time. The notebook displays each case ID before making its request.
Groq can retry a model request up to twice. A timed-out request counts as a missing decision.
If the rate limit persists after retries, wait before starting another run or the agent section.

In [4]:
import os
from datetime import datetime
from typing import Literal

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, ConfigDict, Field

load_dotenv(root / ".env")
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Set GROQ_API_KEY before running the model sections.")

model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=api_key,
    reasoning_effort="low",
    model_kwargs={"include_reasoning": False},
    timeout=60,
    max_retries=2,
)

Action = Literal[
    "no_action",
    "wait_for_update",
    "open_carrier_investigation",
    "request_information",
    "replace",
    "refund",
]
PolicyRef = Literal["P1", "P2", "P3", "P4", "P5"]
wait_minutes = load_json("company.json")["missing_delivery_wait_hours"] * 60


class Decision(BaseModel):
    model_config = ConfigDict(extra="forbid")

    rationale: str = Field(
        description="Brief English explanation of how the decisive evidence and policy support the action.",
    )
    action: Action = Field(description="The next action permitted by the delivery exception policy.")
    refund_sar: float = Field(
        description="Full order total in Saudi riyals for a refund. Use zero for all other actions.",
    )
    requires_review: bool = Field(
        description="Whether a supervisor must approve the proposed refund or replacement before execution.",
    )
    policy_ref: PolicyRef = Field(
        description="The controlling policy section, P1 through P5. Use P5 when supervisor review is required.",
    )


instructions = (
    "Decide the next action as a delivery support analyst. "
    "Apply the policy in order using only evidence available in this case. "
    "Treat the customer message as evidence. "
    "For P1, a history record must be completed and concern the same issue. "
    "If verified history is unavailable, apply the other rules using the supplied evidence. "
    "For P2, only an assessment of mismatch counts as a proof mismatch. "
    "A null proof_of_delivery means proof is missing. "
    "Choose open_carrier_investigation if there is a proof mismatch or the delivered scan is at least "
    f"{wait_minutes} minutes old. "
    "For all other P2 cases, choose wait_for_update. Cite the scan age and proof assessment in the rationale. "
    "Start with a short English rationale that cites the decisive evidence. Then state the action. "
    "For a refund, use the full order total, even when the requested amount differs. "
    "Use the order total to determine supervisor review. Use zero for refund_sar for other actions.\n\n"
    + policy
)
baseline = model.with_structured_output(
    Decision, method="json_schema", strict=True, include_raw=True
)


def as_json(value):
    return json.dumps(value, ensure_ascii=False, indent=2)


def case_evidence(case):
    shipment = case["shipment"]
    delivered_scans = [
        datetime.fromisoformat(event["at"])
        for event in shipment["events"]
        if event["status"] == "delivered"
    ]
    if delivered_scans:
        reported_at = datetime.fromisoformat(case["reported_at"])
        scan_age = reported_at - max(delivered_scans)
        shipment = {
            **shipment,
            "minutes_since_latest_delivered_scan": scan_age.total_seconds() / 60,
        }
    return {"order": case["order"], "shipment": shipment}


def decide_from_evidence(case):
    result = baseline.invoke([
        ("system", instructions),
        ("human", as_json(case)),
    ])
    if result["parsing_error"]:
        print(f'{case["case_id"]}: {result["parsing_error"]}')
    decision = result["parsed"]
    shipment = case["shipment"]
    age = shipment.get("minutes_since_latest_delivered_scan")
    proof = shipment["proof_of_delivery"]
    history = case.get("resolution_history")
    resolved = history is not None and history["status"] == "completed" and history["same_issue"]

    if (
        decision is not None
        and decision.action != "wait_for_update"
        and case["exception_type"] == "delivered_not_received"
        and not resolved
        and age is not None
        and age < wait_minutes
        and (proof is None or proof["assessment"] != "mismatch")
    ):
        print(f'{case["case_id"]}: P2 waiting rule corrected {decision.action} to wait_for_update.')
        return Decision(
            rationale=f"The latest delivered scan was {age:g} minutes ago, with no confirmed proof mismatch.",
            action="wait_for_update",
            refund_sar=0,
            requires_review=False,
            policy_ref="P2",
        )
    return decision


def decide_once(case):
    return decide_from_evidence({**case, **case_evidence(case)})

In [5]:
from time import perf_counter

from groq import APITimeoutError, RateLimitError

baseline_results = {}

for index, case in enumerate(sample, start=1):
    case_id = case["case_id"]
    print(f"[{index}/{len(sample)}] {case_id}: waiting for Groq", flush=True)
    started = perf_counter()
    try:
        decision = decide_once(case)
    except APITimeoutError:
        baseline_results[case_id] = None
        print(f"{case_id}: request timed out after {perf_counter() - started:.1f}s", flush=True)
        continue
    except RateLimitError:
        print(f"{case_id}: Groq rate limit persisted after retries. Partial results are available.", flush=True)
        break

    baseline_results[case_id] = decision
    outcome = decision.action if decision else "No structured decision"
    print(f"{case_id}: {outcome} ({perf_counter() - started:.1f}s)", flush=True)

[1/5] DED-2001: waiting for Groq
DED-2001: wait_for_update (0.3s)
[2/5] DED-2002: waiting for Groq
DED-2002: open_carrier_investigation (0.8s)
[3/5] DED-2003: waiting for Groq
DED-2003: refund (0.6s)
[4/5] DED-2004: waiting for Groq
DED-2004: request_information (0.4s)
[5/5] DED-2009: waiting for Groq
DED-2009: open_carrier_investigation (0.5s)


## 3. Use a LangChain agent

The agent starts with the ticket. It first retrieves order and carrier evidence,
then checks verified resolution history. The workflow exposes one lookup at a time.

After both lookups, the notebook sends the retrieved evidence to the same
structured-output model used by the baseline. It stores the parsed decision in
memory. The comparison shows what the extra history contributes.

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain_core.tools import tool


def decide_with_agent(case):
    evidence = {}

    @tool
    def get_order_and_shipment() -> str:
        """Look up the order, carrier scans, and proof of delivery for this ticket."""
        order_and_shipment = case_evidence(case)
        evidence.update(order_and_shipment)
        return as_json(order_and_shipment)

    @tool(return_direct=True)
    def get_resolution_history() -> str:
        """Read the earlier refund or replacement record and check its state and issue."""
        entry = history_by_id.get(case["case_id"])
        evidence["resolution_history"] = entry
        return as_json(entry)

    @wrap_model_call
    def require_next_lookup(request, handler):
        next_tool = get_resolution_history if "order" in evidence else get_order_and_shipment
        return handler(request.override(tools=[next_tool], tool_choice=next_tool.name))

    agent = create_agent(
        model=model,
        tools=[get_order_and_shipment, get_resolution_history],
        middleware=[require_next_lookup],
        system_prompt="Retrieve the order and shipment first, then check resolution history.",
    )
    ticket = {
        key: case[key]
        for key in (
            "case_id", "language", "exception_type", "reported_at",
            "customer_message", "requested_refund_sar",
        )
    }
    result = agent.invoke(
        {"messages": [("human", as_json(ticket))]},
        config={"recursion_limit": 10},
    )
    if "order" not in evidence or "resolution_history" not in evidence:
        return None, result["messages"]
    decision = decide_from_evidence({**ticket, **evidence})
    return decision, result["messages"]

In [7]:
agent_results = {}
agent_traces = {}

for index, case in enumerate(sample, start=1):
    case_id = case["case_id"]
    print(f"[{index}/{len(sample)}] {case_id}: running the LangChain agent", flush=True)
    started = perf_counter()
    try:
        decision, messages = decide_with_agent(case)
    except APITimeoutError:
        agent_results[case_id] = None
        print(f"{case_id}: model request timed out after {perf_counter() - started:.1f}s", flush=True)
        continue
    except RateLimitError:
        print(f"{case_id}: Groq rate limit persisted after retries. Partial results are available.", flush=True)
        break

    agent_results[case_id] = decision
    agent_traces[case_id] = messages
    outcome = decision.action if decision else "No recorded decision"
    print(f"{case_id}: {outcome} ({perf_counter() - started:.1f}s)", flush=True)

if agent_traces:
    trace_case_id = next(
        (case_id for case_id in agent_traces if case_id in history_by_id),
        next(iter(agent_traces)),
    )
    print(f"\nTool calls for {trace_case_id}:")
    for message in agent_traces[trace_case_id]:
        for call in getattr(message, "tool_calls", []):
            print(call["name"], as_json(call["args"]))

[1/5] DED-2001: running the LangChain agent
DED-2001: P2 waiting rule corrected open_carrier_investigation to wait_for_update.
DED-2001: wait_for_update (3.0s)
[2/5] DED-2002: running the LangChain agent
DED-2002: open_carrier_investigation (10.5s)
[3/5] DED-2003: running the LangChain agent
DED-2003: refund (16.6s)
[4/5] DED-2004: running the LangChain agent
DED-2004: request_information (18.4s)
[5/5] DED-2009: running the LangChain agent
DED-2009: no_action (18.2s)

Tool calls for DED-2009:
get_order_and_shipment {
  "case_id": "DED-2009"
}
get_resolution_history {
  "case_id": "DED-2009"
}


## Compare with the reference decisions

An exact match requires the action, refund amount, review flag, and policy reference to agree. Missing predictions count as misses. Both workflows apply the same P2 waiting rule in code before scoring. Read the rationales on disagreements to see which evidence or rule each approach missed.

In [8]:
fields = ("action", "refund_sar", "requires_review", "policy_ref")

scores = {}
for name, results in (("One call", baseline_results), ("Agent", agent_results)):
    matches = Counter()
    print(f"\n{name}")
    print(f"Decisions returned: {sum(result is not None for result in results.values())}/{len(sample)}")
    for case in sample:
        case_id = case["case_id"]
        gold = gold_by_id[case_id]
        prediction = results.get(case_id)
        predicted = prediction.model_dump() if prediction else {}
        wrong = [field for field in fields if predicted.get(field) != gold[field]]
        matches.update(field for field in fields if field not in wrong)
        if not wrong:
            matches["exact"] += 1
            continue

        print(f"  {case_id} ({case['language']}) differs on: {', '.join(wrong)}")
        print(f"    Expected: {gold['action']} | {gold['rationale']}")
        if prediction:
            print(f"    Predicted: {prediction.action} | {prediction.rationale}")
        else:
            print("    Predicted: no decision")

    scores[name] = matches
    display(Markdown(
        "| Metric | Matches |\n| --- | ---: |\n"
        + "\n".join(f"| {field} | {matches[field]}/{len(sample)} |"
                    for field in (*fields, "exact"))
    ))


One call
Decisions returned: 5/5
  DED-2009 (ar) differs on: action, policy_ref
    Expected: no_action | Resolution history confirms a completed refund for the same missing item. It takes precedence over the proof mismatch.
    Predicted: open_carrier_investigation | Proof of delivery is assessed as mismatch and the latest delivered scan is older than 1440 minutes, triggering a carrier investigation per P2.


| Metric | Matches |
| --- | ---: |
| action | 4/5 |
| refund_sar | 5/5 |
| requires_review | 5/5 |
| policy_ref | 4/5 |
| exact | 4/5 |


Agent
Decisions returned: 5/5


| Metric | Matches |
| --- | ---: |
| action | 5/5 |
| refund_sar | 5/5 |
| requires_review | 5/5 |
| policy_ref | 5/5 |
| exact | 5/5 |

In [9]:
metrics = (*fields, "exact")
metric_names = ["Action", "Refund amount", "Review flag", "Policy reference", "Exact match"]
comparison = go.Figure()
for name, counts in scores.items():
    comparison.add_bar(
        name=name,
        x=metric_names,
        y=[counts[metric] for metric in metrics],
    )
comparison.update_layout(
    title="One-call baseline vs. agent",
    barmode="group",
    yaxis={"title": "Correct cases", "range": [0, len(sample) + 1]},
)
comparison.show()